# MACOG IaC-Eval Kaggle Runner

Runs the paper-style IaC-Eval harness on Kaggle. Enable Internet in the Kaggle notebook settings, run the preflight cell first, then run the v1 smoke cell before v2 or full runs.


In [ ]:
REPO_URL = "https://github.com/png261/MACOG-implement.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/strands-agent"
BIN_DIR = "/kaggle/working/bin"

V1_SMOKE_TASK = "451"
V2_SMOKE_TASK = "aws/task-057"
MODEL = "custom"
MAX_ITERATIONS = 1
TASK_TIMEOUT = 900
INSTALL_CODEBERT_DEPS = False
USE_MINISTACK = False


## Install System Tools


In [ ]:
import os, pathlib, subprocess

def run(cmd, cwd=None, env=None):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, cwd=cwd, env=env or os.environ, check=True)

pathlib.Path(BIN_DIR).mkdir(parents=True, exist_ok=True)
os.environ["PATH"] = BIN_DIR + os.pathsep + os.environ["PATH"]

if not pathlib.Path(f"{BIN_DIR}/terraform").exists():
    run("curl -fsSL -o /kaggle/working/terraform.zip https://releases.hashicorp.com/terraform/1.13.5/terraform_1.13.5_linux_amd64.zip")
    run(f"unzip -o /kaggle/working/terraform.zip -d {BIN_DIR}")

if not pathlib.Path(f"{BIN_DIR}/opa").exists():
    run(f"curl -fsSL -o {BIN_DIR}/opa https://openpolicyagent.org/downloads/latest/opa_linux_amd64_static")
    run(f"chmod +x {BIN_DIR}/opa")

if USE_MINISTACK and not pathlib.Path(f"{BIN_DIR}/go").exists():
    run("curl -fsSL -o /kaggle/working/go.tar.gz https://go.dev/dl/go1.24.4.linux-amd64.tar.gz")
    run("rm -rf /kaggle/working/go")
    run("tar -C /kaggle/working -xzf /kaggle/working/go.tar.gz")
    run(f"ln -sf /kaggle/working/go/bin/go {BIN_DIR}/go")

os.environ["OPA_BIN"] = f"{BIN_DIR}/opa"
os.environ["GOPATH"] = "/kaggle/working/go-path"
os.environ["GOCACHE"] = "/kaggle/working/go-cache"
run("terraform version")
run("opa version")
if USE_MINISTACK:
    run("go version")


## Clone Repo And Install Python Dependencies


In [ ]:
import pathlib, shutil

if pathlib.Path(WORKDIR).exists():
    shutil.rmtree(WORKDIR)
run(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {WORKDIR}")
run("python -m pip install -U pip setuptools wheel", cwd=WORKDIR)
run("python -m pip install -r requirements.txt", cwd=WORKDIR)
run("python -m pip install datasets sacrebleu pandas python-dotenv", cwd=WORKDIR)
if INSTALL_CODEBERT_DEPS:
    run("python -m pip install bert-score torch transformers", cwd=WORKDIR)


## Configure Model Credentials

Add Kaggle Secrets named `CUSTOM_API_KEY`, `CUSTOM_BASE_URL`, and `CUSTOM_MODEL_ID` for the `custom` backend. This cell also writes `/kaggle/working/strands-agent/.env` for tools that load dotenv.


In [ ]:
import os, pathlib, shlex

ENV_KEYS = [
    "CUSTOM_API_KEY", "CUSTOM_BASE_URL", "CUSTOM_MODEL_ID",
    "OPENROUTER_API_KEY", "OPENROUTER_MODEL",
    "ANTHROPIC_API_KEY", "CLAUDE_MODEL_ID",
    "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN", "AWS_DEFAULT_REGION",
]

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ENV_KEYS:
        try:
            value = secrets.get_secret(key)
        except Exception:
            value = None
        if value:
            os.environ[key] = value
except Exception:
    pass

os.environ.setdefault("MACOG_MODEL_BACKEND", "custom")
docker_available = False
if USE_MINISTACK:
    try:
        subprocess.run("docker info", shell=True, check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=20)
        docker_available = True
    except Exception as exc:
        print(f"MiniStack requested, but Docker is not available in this Kaggle runtime: {exc}")
        print("Falling back to structural deploy validation. Enable Docker/accelerator support or run locally for MiniStack.")

DEPLOY_MODE = "sandbox" if USE_MINISTACK and docker_available else "structural"
os.environ["MACOG_DEVOPS_MODE"] = DEPLOY_MODE
if DEPLOY_MODE == "structural":
    os.environ.setdefault("MACOG_EVAL_FAST_SCHEMA", "1")
else:
    os.environ.pop("MACOG_EVAL_FAST_SCHEMA", None)
os.environ.setdefault("OPA_BIN", f"{BIN_DIR}/opa")
os.environ.setdefault("TF_INPUT", "0")
os.environ.setdefault("TF_IN_AUTOMATION", "1")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ.setdefault("GOPATH", "/kaggle/working/go-path")
os.environ.setdefault("GOCACHE", "/kaggle/working/go-cache")

env_path = pathlib.Path(WORKDIR) / ".env"
dotenv_keys = ENV_KEYS + [
    "MACOG_MODEL_BACKEND", "MACOG_DEVOPS_MODE", "MACOG_EVAL_FAST_SCHEMA",
    "OPA_BIN", "TF_INPUT", "TF_IN_AUTOMATION", "GOPATH", "GOCACHE", "AWS_ENDPOINT_URL",
]
env_lines = []
for key in dotenv_keys:
    value = os.environ.get(key)
    if value:
        env_lines.append(f"{key}={shlex.quote(value)}")
env_path.write_text("\n".join(env_lines) + "\n")
env_path.chmod(0o600)
print(f"Deploy mode: {DEPLOY_MODE}; USE_MINISTACK={USE_MINISTACK}; docker_available={docker_available}")
print(f"Wrote {env_path} with {len(env_lines)} keys; secret values are not printed.")

required = ["CUSTOM_API_KEY", "CUSTOM_BASE_URL", "CUSTOM_MODEL_ID"] if MODEL == "custom" else []
missing = [k for k in required if not os.environ.get(k)]
if missing:
    raise RuntimeError(f"Missing Kaggle Secrets/env vars: {missing}")


## Preflight Test First

Run this before any model call. It verifies imports, both datasets, Terraform, OPA, and MiniStack prerequisites when enabled.


In [ ]:
import shutil, subprocess, sys, textwrap

def capture(cmd, cwd=WORKDIR):
    print("$", cmd)
    out = subprocess.check_output(cmd, shell=True, cwd=cwd, text=True, stderr=subprocess.STDOUT)
    print(out[:2000])
    return out

assert shutil.which("terraform"), "terraform is not on PATH"
assert shutil.which("opa") or os.environ.get("OPA_BIN"), "opa is not available"
capture("terraform version")
capture(f"{os.environ.get('OPA_BIN', 'opa')} version")
capture("python -m py_compile eval/dataset.py eval/run_eval.py eval/_worker.py eval/metrics.py eval/models.py agents/reviewer/tools.py ministack.py")
if USE_MINISTACK and DEPLOY_MODE == "sandbox":
    assert shutil.which("docker"), "docker CLI is not on PATH"
    assert shutil.which("go"), "go is not on PATH"
    capture("docker info")
    capture("go version")
    capture("python - <<'PY'\nfrom ministack import MiniStack\nms = MiniStack(health_timeout=90)\nendpoint = ms.start()\nprint(endpoint)\nms.stop()\nPY")

test_code = r'''
from eval.dataset import load_tasks
v1 = load_tasks(dataset="iac-eval-v1", config="default", task_ids=["451"], limit=1)
v2 = load_tasks(dataset="iac-eval-v2", config="default", task_ids=["aws/task-057"], limit=1)
assert len(v1) == 1, "v1 smoke task did not load"
assert len(v2) == 1, "v2 smoke task did not load"
assert v1[0]["rego_intent"], "v1 Rego policy missing"
assert v2[0]["rego_intent"], "v2 Rego policy missing"
print({"v1": {"id": v1[0]["id"], "difficulty": v1[0]["difficulty"], "resources": v1[0]["resources"]}, "v2": {"id": v2[0]["id"], "difficulty": v2[0]["difficulty"], "resources": v2[0]["resources"]}})
'''
subprocess.run([sys.executable, "-c", test_code], cwd=WORKDIR, check=True)
print("Preflight passed. Now run the v1 smoke cell.")


## Smoke: IaC-Eval v1


In [ ]:
run(
    f"python -m eval.run_eval --dataset iac-eval-v1 --models {MODEL} --config default "
    f"--task-id {V1_SMOKE_TASK} --limit 1 --max-iterations {MAX_ITERATIONS} "
    f"--deploy-mode {DEPLOY_MODE} --task-timeout {TASK_TIMEOUT} "
    + ("--use-ministack " if DEPLOY_MODE == "sandbox" else "")
    + "--output eval/results/kaggle_iac_eval_v1_smoke",
    cwd=WORKDIR,
)


## Smoke: IaC-Eval v2


In [ ]:
run(
    f"python -m eval.run_eval --dataset iac-eval-v2 --models {MODEL} --config default "
    f"--task-id {V2_SMOKE_TASK} --limit 1 --max-iterations {MAX_ITERATIONS} "
    f"--deploy-mode {DEPLOY_MODE} --task-timeout {TASK_TIMEOUT} "
    + ("--use-ministack " if DEPLOY_MODE == "sandbox" else "")
    + "--output eval/results/kaggle_iac_eval_v2_smoke",
    cwd=WORKDIR,
)


## Full Runs

Run these only after both smoke tasks pass. Increase `MAX_ITERATIONS` to match the paper setting you want to compare.


In [ ]:
# Full v1: 458 tasks
# run(
#     f"python -m eval.run_eval --dataset iac-eval-v1 --models {MODEL} --config default "
#     f"--max-iterations 3 --deploy-mode {DEPLOY_MODE} --task-timeout 1200 "
#     + ("--use-ministack " if DEPLOY_MODE == "sandbox" else "")
#     + "--output eval/results/kaggle_iac_eval_v1_full",
#     cwd=WORKDIR,
# )

# Full v2: 186 tasks
# run(
#     f"python -m eval.run_eval --dataset iac-eval-v2 --models {MODEL} --config default "
#     f"--max-iterations 3 --deploy-mode {DEPLOY_MODE} --task-timeout 1200 "
#     + ("--use-ministack " if DEPLOY_MODE == "sandbox" else "")
#     + "--output eval/results/kaggle_iac_eval_v2_full",
#     cwd=WORKDIR,
# )


## Results


In [ ]:
run("find eval/results -maxdepth 4 -name 'paper_summary_*.md' -print", cwd=WORKDIR)
